# 🧪 Laboratorio de Implementación · Módulo 3
### Redes Neuronales — Deep Learning · Maestría en Ciencia de Datos · Universidad Santo Tomás

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JotaMao1985/Deep_Learning_Usta/blob/main/notebooks/06-lab-implementacion-modulo-3.ipynb)

Este cuaderno es el **complemento ejecutable** del material del Módulo 3: pasa de **comparar TensorFlow/Keras 3 y PyTorch lado a lado**, por la **aceleración en GPU con precisión mixta**, los **pipelines de datos eficientes** y termina en el **guardado y la exportación a ONNX** lista para desplegar, reproduciendo en código las ideas de los Capítulos 3.1 a 3.3.

> **Cómo usarlo:** activar la GPU antes de ejecutar (en Google Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU (T4)*** → *Guardar*), y luego *Entorno de ejecución → Ejecutar todas*. Está calibrado para correr de punta a punta en una **T4 en ~3–5 minutos** gracias a datos sintéticos y entrenamientos cortos. **También corre sin GPU** (degrada con avisos, no se rompe). Conviene cambiar los valores y volver a ejecutar para observar el efecto.

> ⚠️ **Importante:** este laboratorio usa **datos ligeros y sintéticos** (un *proxy* sintético de imágenes generado con ruido; la carga de datos reales queda comentada) y **no constituye la solución de la Actividad evaluable · Fase 3 «Centinela»**. Su propósito es practicar las herramientas profesionales —frameworks, GPU, pipelines y despliegue— que esa actividad exige.

---
**Contenido**
1. Preparación del entorno y verificación de hardware
2. Frameworks lado a lado: Keras 3 vs PyTorch · *Cap. 3.1*
3. Aceleración en GPU + precisión mixta (AMP) · *Cap. 3.2*
4. Pipelines de datos eficientes y *Data Augmentation* · *Cap. 3.3*
5. Guardado y exportación a ONNX · *Cap. 3.3*
6. Puente a la Actividad (Centinela · Fase 3)

## 1. Preparación del entorno y verificación de hardware

A diferencia de los laboratorios anteriores, este módulo trata de **implementación profesional**: cómo se ejecuta el aprendizaje profundo en hardware real. Por eso lo primero es **detectar y reportar la GPU** disponible. En Colab, tras elegir el entorno *GPU (T4)*, deberían aparecer la tarjeta y su memoria; si no hay GPU, el cuaderno sigue funcionando en CPU con avisos claros.

Se importan los **dos frameworks** del Módulo 3 —PyTorch (siempre) y TensorFlow/Keras (si está disponible)—, se fija una **semilla** para reproducibilidad y se configura la paleta USTA.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 1 · Preparación del entorno y verificación de hardware
# Qué hace · Importa torch y (si existe) tensorflow, fija la semilla y reporta CPU/GPU.
# Por qué  · El Módulo 3 es sobre ejecución real: conviene saber QUÉ hardware nos toca.

# En Colab estas librerías ya están instaladas (torch + tensorflow). Si faltara alguna:
# !pip -q install torch torchvision
# !pip -q install onnx onnxruntime   # se usan en la sección 5 (exportación)

%matplotlib inline
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Reproducibilidad: misma semilla -> mismos resultados
SEMILLA = 42
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# Detección del dispositivo. Este lab PREFIERE GPU (CUDA) porque mide aceleración;
# si no hay, usa MPS (Mac Apple Silicon) o CPU y avisa.
HAY_CUDA = torch.cuda.is_available()
if HAY_CUDA:
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# Paleta USTA para las gráficas
USTA_MORADO, USTA_ROSA, USTA_NAVY = "#3D008D", "#ED1E79", "#001A4D"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

print(f"PyTorch {torch.__version__}")
print(f"Dispositivo de cálculo: {DEVICE}")

# Reporte detallado de la GPU vista por PyTorch
if HAY_CUDA:
    print(f"GPU (PyTorch): {torch.cuda.get_device_name(0)}")
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM total: {total_gb:.1f} GB · CUDA {torch.version.cuda} · cuDNN {torch.backends.cudnn.version()}")
else:
    print("⚠️  No se detectó GPU CUDA. El lab corre igual, pero las secciones de aceleración")
    print("    mostrarán un aviso en vez de un speed-up real. En Colab: Entorno → GPU (T4).")

# TensorFlow / Keras: en Colab vienen instalados; localmente puede faltar -> se envuelve.
try:
    import tensorflow as tf
    HAY_TF = True
    gpus_tf = tf.config.list_physical_devices("GPU")
    print(f"\nTensorFlow {tf.__version__} · Keras {tf.keras.__version__}")
    print(f"GPUs visibles por TensorFlow: {len(gpus_tf)}  {[g.name for g in gpus_tf]}")
except Exception as e:
    HAY_TF = False
    print(f"\n⚠️  TensorFlow no disponible en este entorno ({type(e).__name__}).")
    print("    Las celdas de TF/Keras se omitirán con un aviso. En Colab sí está instalado.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 2 · Reporte de hardware con nvidia-smi
# Qué hace · Ejecuta nvidia-smi para ver la GPU, su VRAM y los procesos en curso.
# Por qué  · Es la herramienta estándar para vigilar la GPU; saber leerla es parte del oficio.

if HAY_CUDA:
    # En Colab/Linux con GPU, nvidia-smi imprime el modelo, la VRAM y la utilización.
    !nvidia-smi
else:
    print("nvidia-smi no aplica: no hay GPU CUDA en este entorno.")
    print("En Colab con GPU (T4) verías aquí la tabla de memoria y utilización de la tarjeta.")


### Datos: un *proxy* sintético de imágenes

Para no descargar conjuntos pesados, se generan imágenes **sintéticas** de 3×64×64 (RGB) en dos clases: **normal** (figura uniforme) y **anómala** (figura con *manchas* contrastantes). Es un campo de práctica con la misma *forma* que la **rama de visión** de la Fase 3, pero totalmente artificial: practica el pipeline sin tocar los datos reales.

> 💡 **De Colab a los datos reales:** la carga de un dataset **real** de imágenes queda **comentada** abajo. En la Fase 3 se sustituye este generador por los datos reales elegidos por el estudiante (no de Kaggle).

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 3 · Generador de un proxy sintético de imágenes (3×64×64)
# Qué hace · Crea N imágenes RGB: clase 0 = figura uniforme, clase 1 = figura con manchas (anomalía).
# Por qué  · Datos ligeros y deterministas: sin descargas, pero con la forma de un problema de visión.

C, H, W = 3, 64, 64          # canales, alto, ancho
N_TRAIN, N_TEST = 480, 120   # subconjuntos pequeños para correr en minutos
CLASES = ["normal", "anómala"]
rng = np.random.default_rng(SEMILLA)

def generar_figuras(n, semilla):
    # Devuelve (X, y): X en [0,1] forma (n,3,64,64); y en {0,1}.
    rg = np.random.default_rng(semilla)
    X = np.zeros((n, C, H, W), dtype=np.float32)
    y = rg.integers(0, 2, size=n)
    yy, xx = np.mgrid[0:H, 0:W]
    for i in range(n):
        # Fondo con textura suave y un color base aleatorio
        base = np.zeros((C, H, W), dtype=np.float32)
        base[0] = 0.20 + 0.10 * rg.random()                     # algo de rojo
        base[1] = 0.40 + 0.15 * rg.random()                     # verde
        base[2] = 0.55 + 0.10 * rg.random()                     # azul dominante
        base += 0.04 * rg.standard_normal((C, H, W)).astype(np.float32)  # textura
        # Figura: un disco centrado (para que no sea un rectángulo perfecto)
        figura = ((xx - W/2)**2 + (yy - H/2)**2) <= (0.40 * W)**2
        base *= figura[None, :, :]
        if y[i] == 1:
            # Clase anómala: varias manchas de color contrastante
            for _ in range(rg.integers(3, 7)):
                cx, cy, r = rg.integers(10, 54), rg.integers(10, 54), rg.integers(4, 9)
                mancha = ((xx - cx)**2 + (yy - cy)**2) <= r**2
                base[0][mancha] = 0.85   # sube rojo/naranja
                base[1][mancha] = 0.30
                base[2][mancha] = 0.10
        X[i] = np.clip(base, 0.0, 1.0)
    return X, y.astype(np.int64)

X_tr, y_tr = generar_figuras(N_TRAIN, SEMILLA)
X_te, y_te = generar_figuras(N_TEST, SEMILLA + 1)
print(f"Entrenamiento: {N_TRAIN} imágenes · Prueba: {N_TEST} imágenes")
print(f"Cada imagen: {C} canales × {H} × {W} · clases: {CLASES}")

# --- Carga de un dataset REAL de imágenes, COMENTADA a propósito ---
# En la Fase 3 se reemplaza este generador por los datos reales elegidos por el estudiante (no de Kaggle):
# !pip -q install kagglehub  # (o descarga directa desde el repositorio institucional declarado)
# from torchvision import datasets, transforms
# ds_real = datasets.ImageFolder("ruta/a/mis/imagenes", transform=transforms.ToTensor())
# (luego: DataLoader sobre ds_real en lugar de generar_figuras)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 4 · Vistazo a los datos sintéticos
# Qué hace · Dibuja una rejilla de 8 imágenes con su etiqueta (normal / anómala).
# Por qué  · Conviene mirar los datos antes de modelar: ¿se distinguen las manchas?

fig, axes = plt.subplots(2, 4, figsize=(8.5, 4.6))
for ax, i in zip(axes.ravel(), range(8)):
    ax.imshow(np.transpose(X_tr[i], (1, 2, 0)))   # (C,H,W) -> (H,W,C) para matplotlib
    color = USTA_NAVY if y_tr[i] == 0 else USTA_ROSA
    ax.set_title(CLASES[y_tr[i]], fontsize=10, color=color)
    ax.axis("off")
fig.suptitle("Proxy sintético de imágenes (navy = normal · rosa = anómala)")
plt.tight_layout(); plt.show()


## 2. Frameworks lado a lado: Keras 3 vs PyTorch · *Cap. 3.1*

Los dos grandes frameworks del ecosistema resuelven el mismo problema con **filosofías distintas**:

- **TensorFlow / Keras 3** privilegia la **declaratividad**: se *describe* la arquitectura con `keras.Sequential` y el entrenamiento se delega a un `model.fit(...)` de alto nivel. Es como dar instrucciones a una **notaría** que se encarga del trámite.
- **PyTorch** privilegia el **control explícito**: se hereda de `nn.Module`, se escribe el `forward` y el bucle de entrenamiento se programa paso a paso (*`zero_grad` → forward → error → `backward` → `step`*). Es como cocinar uno mismo, controlando cada ingrediente.

Aquí se define **la misma CNN pequeña** en ambos frameworks y se comparan **el número de parámetros** y **el estilo**. La arquitectura es idéntica: 2 bloques `conv→ReLU→pool` + capa densa para 2 clases.

> 🌉 **Keras 3 como puente:** Keras 3 puede correr sobre TensorFlow, PyTorch o JAX. La misma `Sequential` cambiaría de motor con solo fijar `KERAS_BACKEND`, sin tocar el código del modelo.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 5 · La MISMA CNN en PyTorch (nn.Module, estilo explícito)
# Qué hace · Define una CNN compacta heredando de nn.Module y cuenta sus parámetros.
# Por qué  · Estilo PyTorch: se escribe el forward y luego un bucle de entrenamiento a mano.

class CNN_PyTorch(nn.Module):
    # CNN compacta para imágenes 3x64x64: 2 bloques conv + densa de 2 salidas.
    def __init__(self, n_clases=2):
        super().__init__()
        self.extractor = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),   # 3 -> 16 canales, conserva 64×64
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 64 -> 32
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 16 -> 32 canales, conserva 32×32
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 32 -> 16
        )
        self.clasificador = nn.Sequential(
            nn.Flatten(),                       # aplana el volumen 32×16×16
            nn.Linear(32 * 16 * 16, n_clases),  # capa densa -> logits de las 2 clases
        )

    def forward(self, x):
        x = self.extractor(x)
        return self.clasificador(x)             # logits (sin softmax: lo aplica la pérdida)

torch.manual_seed(SEMILLA)
cnn_pt = CNN_PyTorch().to(DEVICE)
n_param_pt = sum(p.numel() for p in cnn_pt.parameters())
print(cnn_pt)
print(f"\nParámetros entrenables (PyTorch): {n_param_pt:,}".replace(",", "."))


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 6 · La MISMA CNN en Keras 3 / TensorFlow (Sequential, estilo declarativo)
# Qué hace · Define la idéntica arquitectura con keras.Sequential y cuenta sus parámetros.
# Por qué  · Estilo Keras: se describe la red y model.fit(...) se encarga del entrenamiento.

if HAY_TF:
    from tensorflow import keras
    from tensorflow.keras import layers

    keras.utils.set_random_seed(SEMILLA)
    # OJO: Keras usa el formato channels_last (alto, ancho, canales) -> entrada (64,64,3).
    cnn_tf = keras.Sequential([
        keras.Input(shape=(H, W, C)),
        layers.Conv2D(16, 3, padding="same", activation="relu"),  # -> 64×64×16
        layers.MaxPooling2D(2),                                    # -> 32×32×16
        layers.Conv2D(32, 3, padding="same", activation="relu"),  # -> 32×32×32
        layers.MaxPooling2D(2),                                    # -> 16×16×32
        layers.Flatten(),
        layers.Dense(2),                                           # logits de las 2 clases
    ], name="CNN_Keras")
    cnn_tf.summary()
    n_param_tf = cnn_tf.count_params()
    print(f"\nParámetros entrenables (Keras 3): {n_param_tf:,}".replace(",", "."))
else:
    n_param_tf = None
    print("⚠️  TensorFlow no disponible: se omite la CNN en Keras 3.")
    print("    En Colab esta celda imprimiría el model.summary() con el MISMO número de parámetros")
    print(f"    que la versión PyTorch ({n_param_pt:,} parámetros).".replace(",", "."))


### Comparación de estilo

| Aspecto | **TensorFlow / Keras 3** | **PyTorch** |
|---|---|---|
| Definición del modelo | `keras.Sequential([...])` (declarativo) | `class ...(nn.Module)` + `forward` (explícito) |
| Formato de imagen por defecto | *channels-last* `(H, W, C)` | *channels-first* `(C, H, W)` |
| Entrenamiento | `model.compile(...)` + `model.fit(...)` | bucle a mano: `zero_grad → forward → loss → backward → step` |
| Grafo de cómputo | estático/optimizable (con `@tf.function`) | dinámico («*define-by-run*», río Magdalena) |
| Despliegue nativo | TF Serving · LiteRT · TF.js | TorchServe · `torch.export` · ONNX |
| Curva de aprendizaje | suave para empezar | más control, más código |

Ambas redes tienen **exactamente los mismos parámetros** porque la arquitectura es idéntica: la diferencia está en *cómo se escribe y se entrena*, no en *qué se calcula*. En la práctica se elige según el equipo, el ecosistema de despliegue y la preferencia personal — y **Keras 3** permite incluso compartir motor entre ambos mundos.

## 3. Aceleración en GPU + precisión mixta (AMP) · *Cap. 3.2*

Una GPU es como una **panadería con mil hornos pequeños**: hace muchísimas operaciones a la vez. Entrenar la misma CNN en **CPU vs GPU** muestra el porqué de todo el Módulo 3. Además, la **precisión mixta automática (AMP)** usa números de 16 bits (`float16`) donde es seguro y 32 bits donde hace falta: en los **Tensor Cores** de una T4 esto suele dar **2–3× de aceleración** y **menos uso de VRAM**, sin perder calidad.

Para que el efecto sea visible, se entrena la CNN PyTorch unas pocas iteraciones con datos sintéticos y se **cronometra**. Si no hay GPU, se reporta solo el tiempo de CPU con un aviso.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 7 · Tensores en GPU y utilidad de cronometraje
# Qué hace · Pasa los datos sintéticos a tensores y define una función para medir tiempo.
# Por qué  · Cronometrar bien exige sincronizar la GPU (es asíncrona) antes de tomar el tiempo.

X_tr_t = torch.tensor(X_tr)            # (N,3,64,64) ya en [0,1]
y_tr_t = torch.tensor(y_tr)
EPOCAS_BENCH = 3                       # pocas pasadas: solo queremos comparar tiempos

def entrenar_y_cronometrar(modelo, device, usar_amp=False, epocas=EPOCAS_BENCH):
    # Entrena 'modelo' 'epocas' veces sobre todo el lote y devuelve (segundos, perdida_final).
    modelo = modelo.to(device)
    Xd, yd = X_tr_t.to(device), y_tr_t.to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    # GradScaler solo tiene sentido con AMP en CUDA (escala la pérdida para no perder gradientes)
    usar_amp = usar_amp and (device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=usar_amp)

    if device.type == "cuda":
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    for _ in range(epocas):
        modelo.train()
        opt.zero_grad()
        if usar_amp:
            with torch.autocast("cuda", dtype=torch.float16):  # precisión mixta
                loss = crit(modelo(Xd), yd)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        else:
            loss = crit(modelo(Xd), yd)
            loss.backward(); opt.step()
    if device.type == "cuda":
        torch.cuda.synchronize()
    seg = time.perf_counter() - t0
    return seg, float(loss.item())

print(f"Función de cronometraje lista · {EPOCAS_BENCH} épocas por medición.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 8 · CPU vs GPU vs GPU+AMP: speed-up y memoria
# Qué hace · Entrena la CNN en CPU, en GPU (FP32) y en GPU con precisión mixta (AMP).
# Por qué  · Cuantifica POR QUÉ se usa GPU y AMP: aceleración real y ahorro de VRAM.

resultados = {}

# 1) CPU (referencia, siempre disponible)
torch.manual_seed(SEMILLA)
seg_cpu, _ = entrenar_y_cronometrar(CNN_PyTorch(), torch.device("cpu"))
resultados["CPU (FP32)"] = seg_cpu
print(f"CPU (FP32)        · {seg_cpu:6.3f} s")

if HAY_CUDA:
    # 2) GPU en precisión completa (FP32)
    torch.manual_seed(SEMILLA)
    seg_gpu, _ = entrenar_y_cronometrar(CNN_PyTorch(), torch.device("cuda"), usar_amp=False)
    mem_fp32 = torch.cuda.max_memory_allocated() / 1e6  # MB pico
    resultados["GPU (FP32)"] = seg_gpu
    print(f"GPU (FP32)        · {seg_gpu:6.3f} s · VRAM pico {mem_fp32:6.1f} MB")

    # 3) GPU con precisión mixta (AMP, float16 en Tensor Cores)
    torch.manual_seed(SEMILLA)
    seg_amp, _ = entrenar_y_cronometrar(CNN_PyTorch(), torch.device("cuda"), usar_amp=True)
    mem_amp = torch.cuda.max_memory_allocated() / 1e6
    resultados["GPU + AMP (FP16)"] = seg_amp
    print(f"GPU + AMP (FP16)  · {seg_amp:6.3f} s · VRAM pico {mem_amp:6.1f} MB")

    print(f"\n→ Speed-up GPU vs CPU:        {seg_cpu / seg_gpu:4.1f}×")
    print(f"→ Speed-up GPU+AMP vs GPU:    {seg_gpu / seg_amp:4.1f}×  (los tiempos cortos son ruidosos)")
    print(f"→ Ahorro de VRAM con AMP:     {100*(1 - mem_amp/mem_fp32):4.1f}%")
    print("\nNota: con modelos y lotes pequeños el speed-up es modesto; crece mucho con redes grandes.")
else:
    print("\n⚠️  Sin GPU: solo se midió la CPU. En Colab con T4 verías aquí el speed-up de")
    print("    la GPU (típicamente varias veces más rápida) y el ahorro de VRAM con AMP.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 9 · Gráfico de tiempos (CPU vs GPU vs GPU+AMP)
# Qué hace · Dibuja un diagrama de barras con el tiempo de cada configuración.
# Por qué  · Visualizar el speed-up lo hace tangible; menos barra = más rápido.

etiquetas = list(resultados.keys())
tiempos = [resultados[k] for k in etiquetas]
colores = [USTA_NAVY, USTA_MORADO, USTA_ROSA][:len(etiquetas)]

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(etiquetas, tiempos, color=colores)
ax.set_ylabel(f"Tiempo de {EPOCAS_BENCH} épocas (s)")
ax.set_title("Aceleración del entrenamiento: menos es mejor")
for b, t in zip(barras, tiempos):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f"{t:.2f}s",
            ha="center", va="bottom", fontsize=10)
plt.tight_layout(); plt.show()

if not HAY_CUDA:
    print("(Solo se muestra la barra de CPU porque no hay GPU en este entorno.)")


## 4. Pipelines de datos eficientes y *Data Augmentation* · *Cap. 3.3*

De nada sirve una GPU veloz si se queda **esperando los datos** (*GPU starvation*): es como tener mil hornos y un solo mesero trayendo la masa. Un **pipeline de entrada eficiente** solapa la preparación de datos (en CPU) con el cálculo (en GPU). Las dos cajas de herramientas del módulo:

- **`tf.data`** (declarativo): `dataset.map(...).cache().shuffle(...).batch(...).prefetch(AUTOTUNE)`.
- **`torch.utils.data.DataLoader`** (explícito): `DataLoader(ds, num_workers=..., pin_memory=..., prefetch_factor=...)`.

Ambos persiguen lo mismo: que la GPU **nunca tenga hambre**. Además se ilustra la **Data Augmentation** (volteos, rotaciones, *color jitter*), que multiplica artificialmente los datos y mejora la generalización — **con cuidado**: rotar 180° una imagen puede ser contraproducente si la orientación importa.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 10 · DataLoader de PyTorch (explícito)
# Qué hace · Envuelve los tensores en un Dataset y construye un DataLoader con prefetch.
# Por qué  · num_workers + pin_memory + prefetch_factor evitan que la GPU espere los datos.

from torch.utils.data import TensorDataset, DataLoader

ds_train = TensorDataset(X_tr_t, y_tr_t)

# num_workers: procesos que preparan lotes en paralelo (en CPU). En Colab 2 va bien.
# pin_memory: memoria 'fijada' que acelera la copia CPU->GPU (solo útil con CUDA).
# prefetch_factor: cuántos lotes adelanta cada worker (requiere num_workers > 0).
NUM_WORKERS = 2 if HAY_CUDA else 0
loader_kwargs = dict(batch_size=64, shuffle=True,
                     num_workers=NUM_WORKERS, pin_memory=HAY_CUDA)
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(ds_train, **loader_kwargs)
xb, yb = next(iter(train_loader))
print(f"DataLoader listo · {len(train_loader)} lotes de hasta 64 · num_workers={NUM_WORKERS}")
print(f"pin_memory={HAY_CUDA} · forma de un lote: {tuple(xb.shape)}")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 11 · Pipeline tf.data (declarativo)
# Qué hace · Construye un tf.data.Dataset con map/cache/shuffle/batch/prefetch(AUTOTUNE).
# Por qué  · Es el patrón canónico de TensorFlow para alimentar la GPU sin cuellos de botella.

if HAY_TF:
    import tensorflow as tf
    AUTOTUNE = tf.data.AUTOTUNE

    # Keras espera channels-last: pasamos (N,3,64,64) -> (N,64,64,3)
    X_tr_tf = np.transpose(X_tr, (0, 2, 3, 1)).astype("float32")

    def normaliza(img, etq):
        # Ejemplo de .map: aquí solo asegura el rango [0,1]; podría ir aquí el augmentation.
        return tf.clip_by_value(img, 0.0, 1.0), etq

    ds_tf = (tf.data.Dataset.from_tensor_slices((X_tr_tf, y_tr))
             .map(normaliza, num_parallel_calls=AUTOTUNE)  # transforma en paralelo
             .cache()                                       # guarda en memoria tras la 1ª época
             .shuffle(256, seed=SEMILLA)                    # baraja con un buffer
             .batch(64)                                     # agrupa en mini-lotes
             .prefetch(AUTOTUNE))                           # adelanta el siguiente lote
    lote_img, lote_etq = next(iter(ds_tf))
    print(f"tf.data listo · forma de un lote: {tuple(lote_img.shape)} (channels-last)")
    print("Cadena: from_tensor_slices → map → cache → shuffle → batch → prefetch(AUTOTUNE)")
else:
    print("⚠️  TensorFlow no disponible: se omite el pipeline tf.data.")
    print("    En Colab esta celda construiría el Dataset con map/cache/shuffle/batch/prefetch.")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 12 · Data Augmentation visualizada (antes / después)
# Qué hace · Aplica flip + rotación + color jitter con torchvision.transforms y los grafica.
# Por qué  · El augmentation multiplica los datos; verlo evita aplicar transformaciones dañinas.

from torchvision.transforms import v2 as T

# Transformaciones fotométricas + geométricas (v2: la API recomendada de torchvision).
aumento = T.Compose([
    T.RandomHorizontalFlip(p=1.0),                       # volteo horizontal
    T.RandomRotation(degrees=20),                        # rotación leve (no 180°: cambiaría la orientación)
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),  # variación de color
])

torch.manual_seed(SEMILLA)
base = X_tr_t[:4]                                          # 4 imágenes originales
fig, axes = plt.subplots(2, 4, figsize=(9, 4.8))
for col in range(4):
    orig = base[col]
    aug = aumento(orig)
    axes[0, col].imshow(np.transpose(orig.numpy(), (1, 2, 0))); axes[0, col].axis("off")
    axes[1, col].imshow(np.transpose(aug.clamp(0, 1).numpy(), (1, 2, 0))); axes[1, col].axis("off")
axes[0, 0].set_title("Original", loc="left", fontsize=10, color=USTA_NAVY)
axes[1, 0].set_title("Aumentada", loc="left", fontsize=10, color=USTA_ROSA)
fig.suptitle("Data Augmentation · fila superior: original · fila inferior: transformada")
plt.tight_layout(); plt.show()

print("⚠️  Augmentation con criterio: una rotación de 180° o un flip vertical pueden")
print("    cambiar el significado de la imagen. Elige transformaciones plausibles para tu dominio.")


## 5. Guardado y exportación a ONNX · *Cap. 3.3*

Entrenar es solo la mitad: hay que **guardar el modelo** y, para desplegarlo, **exportarlo** a un formato portable. El módulo distingue:

- **Guardar pesos** para seguir trabajando: PyTorch `torch.save(state_dict)` · Keras `.keras`.
- **Exportar para producción**: **ONNX** (*Open Neural Network Exchange*) es el «**pasaporte**» del modelo — un formato neutro que corre en muchos *runtimes* (servidores, móvil, navegador) sin depender del framework de origen. Para el borde existen además la **cuantización** (pasar a enteros de 8 bits para reducir tamaño/latencia) y **LiteRT** (antes TF Lite).

Aquí se guardan los pesos, se exporta la CNN a ONNX y se **verifica** que `onnxruntime` produce la misma salida que PyTorch con `np.allclose(..., atol=1e-4)`.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 13 · Guardar pesos: PyTorch (state_dict) y Keras (.keras)
# Qué hace · Persiste los pesos del modelo PyTorch y, si hay TF, guarda el modelo Keras.
# Por qué  · Guardar es el paso mínimo para reanudar o desplegar; cada framework tiene su formato.

# PyTorch: lo recomendado es guardar SOLO el state_dict (los pesos), no el objeto entero.
torch.save(cnn_pt.state_dict(), "cnn_demo.pth")
print("PyTorch · pesos guardados en 'cnn_demo.pth' (state_dict).")

# Reconstrucción: crear la arquitectura y cargar los pesos.
# weights_only=True (seguro): evita ejecutar código arbitrario al deserializar.
cnn_recargada = CNN_PyTorch().to(DEVICE)
cnn_recargada.load_state_dict(torch.load("cnn_demo.pth", weights_only=True))
cnn_recargada.eval()
print("PyTorch · pesos recargados en una arquitectura nueva (weights_only=True, seguro).")

# Keras 3: el formato moderno es un único archivo .keras (arquitectura + pesos).
if HAY_TF:
    cnn_tf.save("cnn_demo.keras")
    print("Keras 3 · modelo guardado en 'cnn_demo.keras' (formato nativo).")
else:
    print("⚠️  TF no disponible: se omite el guardado .keras (en Colab: cnn_tf.save('cnn_demo.keras')).")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Ventana 14 · Exportar a ONNX y verificar la inferencia
# Qué hace · Exporta la CNN PyTorch a ONNX y compara su salida con onnxruntime (np.allclose).
# Por qué  · ONNX es el 'pasaporte' del modelo; verificar la paridad evita sorpresas al desplegar.

# onnx y onnxruntime suelen estar en Colab; si faltan, se instalan en silencio.
try:
    import onnx, onnxruntime  # noqa: F401
    HAY_ONNX = True
except Exception:
    print("Instalando onnx + onnxruntime ...")
    import subprocess, sys
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                          "onnx", "onnxruntime", "onnxscript"])
    try:
        import onnx, onnxruntime  # noqa: F401
        HAY_ONNX = True
    except Exception as e:
        HAY_ONNX = False
        print(f"⚠️  No se pudo instalar onnxruntime ({type(e).__name__}). Se omite la exportación.")

if HAY_ONNX:
    import onnxruntime as ort
    # La exportación a ONNX es más estable en CPU; el modelo es diminuto, así que copiamos ahí.
    cnn_cpu = CNN_PyTorch()
    cnn_cpu.load_state_dict(cnn_pt.state_dict())
    cnn_cpu.eval()
    ejemplo = torch.tensor(X_te[:1])                      # 1 imagen de prueba (1,3,64,64)

    try:
        # dynamo=False fuerza el exportador estable (TorchScript), que solo necesita 'onnx'.
        # En versiones antiguas de torch este argumento no existe -> se reintenta sin él.
        def exportar(**extra):
            torch.onnx.export(
                cnn_cpu, ejemplo, "cnn_demo.onnx",
                input_names=["entrada"], output_names=["logits"],
                dynamic_axes={"entrada": {0: "lote"}, "logits": {0: "lote"}},
                opset_version=17, **extra,
            )
        try:
            exportar(dynamo=False)
        except TypeError:
            exportar()                                    # torch antiguo: sin el kwarg dynamo
        print("Exportado a 'cnn_demo.onnx'.")

        # Salida de referencia (PyTorch) vs salida del runtime ONNX
        with torch.no_grad():
            salida_pt = cnn_cpu(ejemplo).numpy()
        sesion = ort.InferenceSession("cnn_demo.onnx", providers=["CPUExecutionProvider"])
        salida_onnx = sesion.run(None, {"entrada": ejemplo.numpy()})[0]

        iguales = np.allclose(salida_pt, salida_onnx, atol=1e-4)
        print(f"\nSalida PyTorch : {np.round(salida_pt, 4)}")
        print(f"Salida ONNX    : {np.round(salida_onnx, 4)}")
        print(f"¿Coinciden (atol=1e-4)?  {'✅ sí — el pasaporte es válido' if iguales else '❌ no'}")
    except Exception as e:
        # El exportador moderno (dynamo) puede pedir 'onnxscript'; si falla, no rompemos el lab.
        print(f"⚠️  No se pudo exportar a ONNX en este entorno ({type(e).__name__}: {e}).")
        print("    Suele resolverse con: !pip install onnxscript   (en Colab la exportación funciona).")
else:
    print("Se omitió la exportación a ONNX por falta de runtime (en Colab funciona).")


> 📦 **Para el borde (Edge):** además de ONNX, en despliegues móviles se aplica **cuantización** —convertir los pesos de `float32` a enteros de 8 bits (`int8`)— para reducir el tamaño del modelo y la latencia, a cambio de una pérdida mínima de precisión. En el ecosistema TensorFlow esto se hace con **LiteRT** (antes TF Lite); en PyTorch, con `torch.export` + ExecuTorch o exportando a ONNX y cuantizando con `onnxruntime`. La Fase 3 exige justamente un despliegue **offline** que corra en el dispositivo.

## 6. Puente a la Actividad · Centinela · Fase 3

Lo practicado aquí es **exactamente** el andamiaje técnico de la actividad evaluable de la Fase 3 — aplicado a datos sintéticos y entrenamientos cortos. La Fase 3 pide llevar un modelo a un **pipeline reproducible, acelerado en GPU y desplegable offline**; la tabla traduce cada habilidad ensayada en el laboratorio a su lugar en el proyecto:

| En este laboratorio (datos toy/sintéticos) | En la Fase 3 «Centinela» (datos reales del estudiante) |
|---|---|
| la misma CNN en Keras 3 y PyTorch (Cap. 3.1) | elegir y **justificar un framework** para el modelo definitivo del proyecto |
| CPU vs GPU + precisión mixta (Cap. 3.2) | entrenar en **GPU (Colab/Kaggle)** con **AMP** y *checkpointing* a Drive |
| `DataLoader` / `tf.data` + augmentation (Cap. 3.3) | **pipeline de datos reproducible y eficiente** sobre los datos reales, con augmentation con criterio |
| `torch.save` / `.keras` + export a **ONNX** verificado (Cap. 3.3) | **exportar y desplegar** el modelo (ONNX/LiteRT) y verificar la **paridad de inferencia** |
| mención de cuantización / LiteRT para el borde | **despliegue offline** que corra en el dispositivo (en el borde) |

> ⚠️ **Recordatorio:** este cuaderno **no** entrega la solución. La Fase 3 exige aplicar todo esto a **datos reales elegidos por el estudiante** (no de Kaggle), construir el pipeline completo, acelerar el entrenamiento, lograr un despliegue **offline** funcional, documentar el **uso responsable de IA** (bitácora + auditoría) y sustentar el trabajo oralmente.

> 📝 *Nota de escalado:* aquí se usaron datos sintéticos, lotes pequeños y 3 épocas para correr en minutos. Con conjuntos reales conviene: cargar por lotes desde disco (`ImageFolder` + `DataLoader` / `image_dataset_from_directory`), aumentar las épocas con *early stopping*, vigilar la **VRAM** (regla práctica: la memoria escala con el tamaño del lote y del modelo), guardar *checkpoints* periódicos y medir la latencia del modelo exportado en el dispositivo de destino.

### 🔗 Relacionado
- [Material interactivo del Módulo 3](https://jotamao1985.github.io/Deep_Learning_Usta/) (Capítulos 3.1–3.3: Frameworks · HPC/GPU · Pipelines y Despliegue) y la actividad navegable de la **Fase 3 «Centinela»**.
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3.ª ed.), caps. de despliegue y aceleración. O'Reilly.
- Stevens, E., Antiga, L. & Viehmann, T. (2020). *Deep Learning with PyTorch*. Manning.
- Documentación oficial: [`tf.data`](https://www.tensorflow.org/guide/data) · [`torch.export` / ONNX](https://pytorch.org/docs/stable/onnx.html) · [Keras 3 multi-backend](https://keras.io/keras_3/).